In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import nltk
import re
import keras
from tensorflow.keras.utils import to_categorical
from keras.layers import LSTM,Dense,Dropout
from keras.optimizers import Adamax
from keras.models import Sequential

In [2]:
data = pd.read_csv("Songs.csv")
data.head()

,Artist,Title,Lyrics
0,Taylor Swift,cardigan,"Vintage tee, brand new phone\nHigh heels on co..."
1,Taylor Swift,exile,"I can see you standing, honey\nWith his arms a..."
2,Taylor Swift,Lover,We could leave the Christmas lights up 'til Ja...
3,Taylor Swift,the 1,"I'm doing good, I'm on some new shit\nBeen say..."
4,Taylor Swift,Look What You Made Me Do,I don't like your little games\nDon't like you...


In [3]:
#printing the name of artists
print("Artists in the data:\n",data.Artist.value_counts())

Artists in the data:
 Artist
Taylor Swift     50
Billie Eilish    50
Name: count, dtype: int64


In [4]:
#size of dataset
data.shape

(100, 3)

In [5]:
#display the random song lyrics
data.Lyrics[1][:100]

"I can see you standing, honey\nWith his arms around your body\nLaughin', but the joke's not\u2005funny\u2005at a"

In [6]:
#Lining up all the lyrics to create corpus
corpus =''
for listitem in data.Lyrics:
    corpus += listitem

corpus = corpus.lower()
print("Number of unique characters:", len(set(corpus)))

Number of unique characters: 59


In [7]:
print(sorted(set(corpus)))

['\n', ' ', '!', '"', "'", '(', ')', ',', '-', '.', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '?', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', 'à', 'é', 'í', 'ó', 'е', '\u2005', '—', '‘', '’', '…', '\u205f', '\ufeff']


In [8]:
#Removing unnecessary characters
to_remove = ['{', '}', '~', '©', 'à', 'á', 'ã', 'ä', 'ç', 'è', 'é', 'ê', 'ë', 'í', 'ñ', 'ó', 'ö', 'ü', 'ŏ',
             'е', 'ا', 'س', 'ل', 'م', 'و', '\u2005', '\u200a', '\u200b', '–', '—', '‘', '’', '‚', '“', '”',
             '…', '\u205f', '\ufeff', '!', '&', '(', ')', '*', '-',  '/', ]
for symbol in to_remove:
    corpus = corpus.replace(symbol," ")

In [9]:
symb = sorted(list(set(corpus)))

L_corpus = len(corpus)
L_symb = len(symb)

#Building dictionary to access the vocabulary from indices and vice versa
mapping = dict((c, i) for i, c in enumerate(symb))
reverse_mapping = dict((i, c) for i, c in enumerate(symb))

print("Total number of characters:", L_corpus)
print("Number of unique characters:", L_symb)

Total number of characters: 154632
Number of unique characters: 43


In [10]:
length = 40
features = []
targets = []
for i in range(0, L_corpus - length, 1):
    feature = corpus[i:i + length]
    target = corpus[i + length]
    features.append([mapping[j] for j in feature])
    targets.append(mapping[target])


L_datapoints = len(targets)
print("Total number of sequences in the Corpus:", L_datapoints)

Total number of sequences in the Corpus: 154592


In [11]:
# reshape X and normalize
x = (np.reshape(features, (L_datapoints, length, 1)))/ float(L_symb)
y = to_categorical(targets)

In [12]:
model = Sequential()
#Adding layers
model.add(LSTM(256, input_shape=(x.shape[1], x.shape[2])))
model.add(Dense(y.shape[1], activation='softmax'))
#Compiling the model
opt = Adamax(learning_rate=0.01)
model.compile(loss='categorical_crossentropy', optimizer=opt)

model.summary()

C:\Users\Shaik Irfan\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                          │ (None, 256)                 │         264,192 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 43)                  │          11,051 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 275,243 (1.05 MB)

 Trainable params: 275,243 (1.05 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history = model.fit(x, y, batch_size=128, epochs=50)

Epoch 1/50
 569/1208 ━━━━━━━━━━━━━━━━━━━━ 1:49 172ms/step - loss: 3.0000

In [ ]:
def Lyrics_Generator(starter,Ch_count):
    generated= ""
    starter = starter
    seed=[mapping[char] for char in starter]
    generated += starter

    for i in range(Ch_count):
        seed=[mapping[char] for char in starter]
        x_pred = np.reshape(seed, (1, len(seed), 1))
        x_pred = x_pred/ float(L_symb)
        prediction = model.predict(x_pred, verbose=0)[0]


        prediction = np.asarray(prediction).astype('float64')
        prediction = np.log(prediction) / 1.0
        exp_preds = np.exp(prediction)
        prediction = exp_preds / np.sum(exp_preds)
        probas = np.random.multinomial(1, prediction, 1)
        index = np.argmax(prediction)
        next_char = reverse_mapping[index]


        generated += next_char
        starter = starter[1:] + next_char

    return generated

In [ ]:
song_2 = Lyrics_Generator(
    "love is in the air tonight and ",
    400)
print(song_2)